# Basic Financial Retrieval System

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PDF_PATH, CHUNK_SIZE, CHUNK_OVERLAP
from src.pdf_loader import load_pdf
from src.chunker import chunk_documents

documents = load_pdf(PDF_PATH)
chunks = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Loaded pages: {len(documents)}")
print(f"Created chunks: {len(chunks)}")
print(chunks[0].page_content[:500])
print(chunks[0].metadata)

Loaded pages: 80
Created chunks: 358
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐    TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California 94-
{'producer': 'Wdesk Fidelity Content Translations Version 014.006.087', 'creator': 'Workiva', 'creationdate': '2025-10-30T14:45:03+00:00', 'author': 'anonymous', 'moddate': '2025-10-30T08:52:59-07:00', 'subject': '10-K 2025, 09.27.2025', 'title': '_10-K 2025 (As-Filed)', 'source': 'C:\\AHMEDPC\\Project & Courses\\RAG\\data\\raw\\apple_10k.pdf', 'total_pages': 80, 'page': 0, 'page_label': '1'}


In [2]:
from src.vector_store import build_vector_store
from src.embeddings import get_embedding_model
from src.retriever import retrieve_relevant_chunks
from src.config import (
    EMBEDDING_MODEL_NAME,
    CHROMA_DIR,
    CHROMA_COLLECTION_NAME,
    RETRIEVAL_K,
)

embedding_model = get_embedding_model(EMBEDDING_MODEL_NAME)
vector_store = build_vector_store(
    chunks=chunks,
    embedding_model=embedding_model,
    persist_directory=CHROMA_DIR,
    collection_name=CHROMA_COLLECTION_NAME
)

question = "What were Apple's main sources of revenue?"

retrieved = retrieve_relevant_chunks(
    vector_store=vector_store,
    question=question,
    k=RETRIEVAL_K
)

for i, doc in enumerate(retrieved, start=1):
    print("-" * 80)
    print("chunk ", i)
    print("metadata ", doc.metadata)
    print(doc.page_content[:200])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--------------------------------------------------------------------------------
chunk  1
metadata  {'page': 31, 'total_pages': 80, 'subject': '10-K 2025, 09.27.2025', 'source': 'C:\\AHMEDPC\\Project & Courses\\RAG\\data\\raw\\apple_10k.pdf', 'creator': 'Workiva', 'moddate': '2025-10-30T08:52:59-07:00', 'producer': 'Wdesk Fidelity Content Translations Version 014.006.087', 'creationdate': '2025-10-30T14:45:03+00:00', 'author': 'anonymous', 'title': '_10-K 2025 (As-Filed)', 'page_label': '32'}
Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of shares, which are reflected in thousands, and per-share amounts)
Years ended
September 27,
2025
September 28,
2024
Septe
--------------------------------------------------------------------------------
chunk  2
metadata  {'creationdate': '2025-10-30T14:45:03+00:00', 'moddate': '2025-10-30T08:52:59-07:00', 'total_pages': 80, 'author': 'anonymous', 'subject': '10-K 2025, 09.27.2025', 'producer': 'Wdesk Fidelity Content T

In [6]:
from src.rag_pipeline import answer_question

question = "What were Apple's main sources of revenue?"
result = answer_question(
    question=question,
    vector_store=vector_store,
    k=RETRIEVAL_K
)

print("Answer:", result["answer"])

Answer: content="Apple's main sources of revenue are from the sale of Products and Services [Source 1, Source 2, Source 3].\n\nFor the year ended September 27, 2025:\n*   **Products** generated $307,003 million in net sales [Source 1, Source 2, Source 3]. This represents revenue from the sale of physical goods.\n*   **Services** generated $109,158 million in net sales [Source 1, Source 2, Source 3]. This represents revenue from services offered by the company." additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e60de-57f4-7443-8e76-61a7cf8c5277-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1798, 'output_tokens': 253, 'total_tokens': 2051, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 125}}


In [7]:
for i, doc in enumerate(result["sources"], start=1):
    print("=" * 80)
    print(f"Source {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:500])

Source 1
Metadata: {'total_pages': 80, 'creator': 'Workiva', 'page_label': '32', 'moddate': '2025-10-30T08:52:59-07:00', 'title': '_10-K 2025 (As-Filed)', 'author': 'anonymous', 'page': 31, 'source': 'C:\\AHMEDPC\\Project & Courses\\RAG\\data\\raw\\apple_10k.pdf', 'creationdate': '2025-10-30T14:45:03+00:00', 'producer': 'Wdesk Fidelity Content Translations Version 014.006.087', 'subject': '10-K 2025, 09.27.2025'}
Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of shares, which are reflected in thousands, and per-share amounts)
Years ended
September 27,
2025
September 28,
2024
September 30,
2023
Net sales:
   Products $ 307,003 $ 294,866 $ 298,085 
   Services  109,158  96,169  85,200 
Total net sales  416,161  391,035  383,285 
Cost of sales:
   Products  194,116  185,233  189,282 
   Services  26,844  25,119  24,855 
Total cost of sales  220,960  210,352  214,137 
Gross marg
Source 2
Metadata: {'producer': 'Wdesk Fidelity Content Translations Version 014.0

In [8]:
question = "What is Tim Cook's favorite food?"

result = answer_question(
    question=question,
    vector_store=vector_store,
    k=4,
)

print(result["answer"])

content='I could not find enough information in the retrieved report excerpts to answer this confidently. The provided context only lists Timothy D. Cook as the Chief Executive Officer and Director who signed the report on October 31, 2025 [Source 1, 2, 3, 4]. It does not contain any information about his personal preferences, such as his favorite food.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e60de-e27d-73b1-bcf0-4d64bff4f931-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1527, 'output_tokens': 239, 'total_tokens': 1766, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 161}}
